In [ ]:
import pandas as pd

df = pd.read_csv('C:\\RS_Project\\podcasts_data.csv')
df.head()

In [ ]:
def search_podcasts(query):
    """Return podcasts whose title, author, or category match the query."""
    q = str(query or "").strip().lower()
    if q == "":
        return df.copy()

    mask = df.apply(
        lambda row: q in str(row.get('title', '')).lower()
        or q in str(row.get('author', '')).lower()
        or q in str(row.get('category', '')).lower(),
        axis=1,
    )

    return df[mask]


def recommend_for_category(category, limit=10, exclude_ids=None):
    """Recommend podcasts from the same category as the clicked business tab."""
    if exclude_ids is None:
        exclude_ids = []

    category = str(category or "").strip().lower()
    if category == "":
        return pd.DataFrame([])

    filtered = df[df['category'].str.lower() == category]
    if exclude_ids:
        filtered = filtered[~filtered['id'].astype(str).isin([str(x) for x in exclude_ids])]

    return filtered.head(limit)


def recommend_related(clicked_id, limit=5, exclude_ids=None):
    """When a podcast or business item is clicked, return other podcasts in the same category."""
    clicked_id = str(clicked_id or "").strip()
    if clicked_id == "":
        return pd.DataFrame([])

    if exclude_ids is None:
        exclude_ids = []

    matched = df[df['id'].astype(str) == clicked_id]
    if matched.empty:
        return pd.DataFrame([])

    category = str(matched.iloc[0]['category']).strip().lower()
    related = df[
        (df['category'].str.lower() == category)
        & (df['id'].astype(str) != clicked_id)
    ]

    if exclude_ids:
        related = related[~related['id'].astype(str).isin([str(x) for x in exclude_ids])]

    return related.head(limit)


def format_results(result_df):
    return result_df.to_dict(orient='records')


# Example usage:
print('Search for "Business":')
print(format_results(search_podcasts('Business')))

print('\nRecommend for category "Technology":')
print(format_results(recommend_for_category('Technology')))

print('\nRelated podcasts after clicking item id "3" (Business):')
print(format_results(recommend_related('3')))